In [1]:
!pip -q install transformers datasets accelerate

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments

In [3]:
from huggingface_hub import login

login("")

In [4]:
from datasets import load_dataset
dataset = load_dataset("uitnlp/vigoemotions")

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

val.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/16531 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2067 [00:00<?, ? examples/s]

In [5]:
import re

NUM_LABELS = 28  # ViGoEmotions

def parse_labels(label_str):
    indices = list(map(int, re.findall(r"\d+", label_str)))
    multi_hot = [0.0] * NUM_LABELS
    for idx in indices:
        multi_hot[idx] = 1.0
    return multi_hot

def preprocess(example):
    example["labels"] = parse_labels(example["labels"])
    return example

dataset = dataset.map(preprocess)

Map:   0%|          | 0/16531 [00:00<?, ? examples/s]

Map:   0%|          | 0/2066 [00:00<?, ? examples/s]

Map:   0%|          | 0/2067 [00:00<?, ? examples/s]

In [6]:
class ViGoDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

dataset_to_train = ViGoDataset(dataset['train']['text'], dataset['train']['labels'])

In [25]:
print(dataset_to_train.labels[0])

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])


In [8]:
model_name = "uitnlp/visobert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def collate_fn(batch):
    texts, labels = zip(*batch)
    enc = tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    labels = torch.stack(labels)
    return enc, labels

In [9]:
class Encoder(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.proj = nn.Linear(self.backbone.config.hidden_size, 256)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]  # CLS token
        z = self.proj(cls)
        z = F.normalize(z, dim=1)  # cosine space
        return z

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
#model = Encoder(model_name).to(device)

cuda


In [10]:
def label_similarity(y):
    # y: [B, 28]
    y = F.normalize(y, dim=1)
    sim = torch.matmul(y, y.T)  # [B, B]
    sim.fill_diagonal_(0)       # ignore self
    return sim

In [11]:
def soft_infonce(z, y, temperature=0.1):
    # z: [B, D], y: [B, 28]
    sim_z = torch.matmul(z, z.T) / temperature  # [B, B]
    sim_y = label_similarity(y)                 # weights

    mask = torch.eye(z.size(0), device=z.device).bool()
    sim_z = sim_z.masked_fill(mask, -1e9)

    log_prob = F.log_softmax(sim_z, dim=1)

    # normalize weights per anchor
    weights = sim_y / (sim_y.sum(dim=1, keepdim=True) + 1e-8)

    loss = -(weights * log_prob).sum(dim=1).mean()
    return loss

In [39]:
class ContrastiveTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # inputs is a tuple: (enc_dict, labels)
        enc, labels = inputs

        enc = {k: v for k, v in enc.items()}

        z = model(**enc)
        loss = soft_infonce(z, labels)

        return (loss, z) if return_outputs else loss

In [13]:
model = Encoder(model_name)

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=32,
    num_train_epochs=7,
    learning_rate=2e-5,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    report_to="none"  # change to "wandb" if needed
)

pytorch_model.bin:   0%|          | 0.00/390M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/390M [00:00<?, ?B/s]

In [14]:
model.to(device)

Encoder(
  (backbone): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(15004, 768, padding_idx=1)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (L

In [41]:
trainer = ContrastiveTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_to_train,
    data_collator=collate_fn
)

In [42]:
trainer.train()

Step,Training Loss
50,4.042999
100,3.936610
150,3.864464
200,3.787957
250,3.771430
300,3.634344
350,3.697114
400,3.641930
450,3.661273
500,3.631801


TrainOutput(global_step=1813, training_loss=3.4456142692755285, metrics={'train_runtime': 1233.5793, 'train_samples_per_second': 93.806, 'train_steps_per_second': 1.47, 'total_flos': 0.0, 'train_loss': 3.4456142692755285, 'epoch': 7.0})

In [45]:
import numpy as np
import pandas as pd
from tqdm import tqdm

# -------------------------
# 1. Encode function
# -------------------------
def encode_texts(texts, batch_size=64):
    embs = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            z = model(**enc)

        embs.append(z.cpu())

    return torch.cat(embs, dim=0)


# -------------------------
# 2. Build train embeddings index
# -------------------------
train_ids = dataset['train']["id"]
train_texts = dataset['train']["text"]

train_embs = encode_texts(train_texts)  # [N_train, dim]
train_embs = torch.nn.functional.normalize(train_embs, dim=1)


# -------------------------
# 3. Retrieval function
# -------------------------
def retrieve(query_emb, topk=5):
    sims = torch.matmul(train_embs, query_emb.unsqueeze(1)).squeeze(1)

    most_sim_idx = torch.topk(sims, topk).indices.tolist()
    most_diff_idx = torch.topk(-sims, topk).indices.tolist()

    return most_sim_idx, most_diff_idx


# -------------------------
# 4. Process dev + test
# -------------------------
def process_split(df):
    results = []

    texts = df["text"]
    ids = df["id"]

    query_embs = encode_texts(texts)
    query_embs = torch.nn.functional.normalize(query_embs, dim=1)

    for i in tqdm(range(len(df))):
        sim_idx, diff_idx = retrieve(query_embs[i])

        results.append({
            "query_id": ids[i],
            "similar_ids": [train_ids[j] for j in sim_idx],
            "different_ids": [train_ids[j] for j in diff_idx]
        })

    return results


# -------------------------
# 5. Run dev + test
# -------------------------
dev_results = process_split(dataset['validation'])
test_results = process_split(dataset['test'])

all_results = dev_results + test_results


# -------------------------
# 6. Save CSV
# -------------------------
out_df = pd.DataFrame(all_results)
out_df.to_csv("retrieval_results.csv", index=False)

100%|██████████| 2067/2067 [00:04<00:00, 433.45it/s]
